In [24]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import multiprocessing as mp

import os
import re
import itertools
import pickle
import json
import datetime

from joblib import Parallel, delayed
from tqdm.notebook import tqdm
from collections import defaultdict
from multiprocess import Pool
from scipy import sparse
from scipy.sparse import linalg
from itertools import islice

## Load interactions

In [3]:
df_int = pd.read_csv("../../dataset/final_dataset/contacted_anon.csv")

# Only keep vacancies with at least 15 interactions
relevant_vacancies = df_int["cvid"].value_counts()[df_int["cvid"].value_counts() >= 15].index

# Filter to only relevant vacancies
df_int = df_int[df_int["cvid"].isin(relevant_vacancies.values)][["humanjobid", "response", "cvid"]].fillna(0)
df_int["response"] = df_int["response"].apply(lambda x: x if x == 0 else 1)

non_zero = df_int.groupby("cvid")["response"].sum() > 0
non_zero = set(non_zero[non_zero == True].index)
len(non_zero), len(df_int.groupby("cvid"))

df_graphs = df_int[df_int["cvid"].isin(non_zero)]
df_bridges = pd.read_excel("../outputs/final_outputs/bridges.xlsx")

todo = df_bridges.dropna(subset="bridges_qwen_structured")[["humanjobid", "cvid"]].values

## Load KG

In [38]:
def load_KG(model, prompt, isco=False):

    G = nx.DiGraph()

    print("Loading", model, prompt, isco)    

    if isco:
        filename = f"../outputs/inferred_outputs/kg_{model}_{prompt}_isco.edgelist"
    else:
        filename = f"../outputs/inferred_outputs/kg_{model}_{prompt}.edgelist"
    
    with open(filename, encoding="utf-8") as f:
        for line in tqdm(f.readlines()):
            
            if "err:error" in line:
                print(line)
                continue
                
            s, p, o = eval(line)
            G.add_edge(s, o, edge_type=p)
    
    # Find neighbors of each node (for quicker retrieval later on) 
    all_neighbors = {}
    
    H = G.to_undirected()
    
    for n in tqdm(G.nodes):
        all_neighbors[n] = set(H.neighbors(n))
        
    all_neighbors = {k: {i for i in v if not pd.isna(i)} for k, v in all_neighbors.items()}
    
    # Do this once, outside your pair-loop
    degrees = dict(H.degree())
    weights = {}
    
    for u, v in H.edges():
        # Pre-calculating degrees into a dict is much faster than calling H.degree(u)
        weights[(u, v)] = 1.0 / np.log1p(degrees[u] + degrees[v])
    
    nx.set_edge_attributes(H, weights, 'ppr_weight')

    return G

# BFS

In [50]:
def extract_subgraphs(G_original, H_safe, node_to_comp, degrees, source, target):
    null_metrics = {
        "path_exists": False, "num_nodes": 0, "num_edges": 0,
        "density": 0.0, "avg_degree": 0.0, "avg_clustering": 0.0,
        "shortest_path_length": -1
    }

    if source not in H_safe or target not in H_safe: return None, null_metrics
    if node_to_comp.get(source) != node_to_comp.get(target): return None, null_metrics
    if source == target: return None, null_metrics
        
    final_nodes = {source, target}
    
    try:
        spine_path = nx.bidirectional_shortest_path(H_safe, source, target)
        final_nodes.update(spine_path)
        
        S_neighbors = set(H_safe[source])
        T_neighbors = set(H_safe[target])
        
        common_2hop = S_neighbors.intersection(T_neighbors)
        final_nodes.update(list(common_2hop)[:10])
        
        if len(spine_path) >= 3:
            node_after_source = spine_path[1]
            node_before_target = spine_path[-2]
            
            S_to_NBT = S_neighbors.intersection(set(H_safe[node_before_target]))
            final_nodes.update(list(S_to_NBT)[:5])
            
            NAS_to_T = set(H_safe[node_after_source]).intersection(T_neighbors)
            final_nodes.update(list(NAS_to_T)[:5])

        for seed in [source, target]:
            neighbors = list(H_safe[seed])
            sorted_n = sorted(neighbors, key=lambda x: degrees.get(x, 0))
            final_nodes.update(sorted_n[:10]) 
            
        for n in spine_path[1:-1]:
            if degrees.get(n, 0) < 50:
                final_nodes.update(list(H_safe[n])[:3])

        sub = G_original.subgraph(final_nodes).copy()
        sub.remove_edges_from(nx.selfloop_edges(sub))
        
        # pruning = True
        # while pruning:
        #     pruning = False
        #     leaves = [n for n, d in sub.degree() if d <= 1 and n not in {source, target}]
        #     if leaves:
        #         sub.remove_nodes_from(leaves)
        #         pruning = True 
                
        if source not in sub or target not in sub:
            return None, null_metrics
            
        metrics = dict(null_metrics)
        
        sub_undir = sub.to_undirected()
        has_valid_path = nx.has_path(sub_undir, source, target) 
        metrics["path_exists"] = has_valid_path
        sub_metric_safe = nx.Graph(sub_undir)
        
        v_count = sub_metric_safe.number_of_nodes()
        e_count = sub_metric_safe.number_of_edges()
        
        metrics["num_nodes"] = v_count
        metrics["num_edges"] = e_count
        
        if v_count > 1:
            metrics["density"] = nx.density(sub_metric_safe)
            metrics["avg_degree"] = (2.0 * e_count) / v_count
            metrics["avg_clustering"] = nx.average_clustering(sub_metric_safe)
            
        if has_valid_path:
            # Fix: Calculate path length on the UNDIRECTED graph
            metrics["shortest_path_length"] = nx.shortest_path_length(sub_undir, source, target)

        sub.add_edge(source, target)            
        return sub, metrics
        
    except nx.NetworkXNoPath:
        print("Error: no path exists")
        return None, null_metrics
    except Exception as e:
        print(f"\nError on {source} -> {target}: {e}")
        return None, null_metrics

def sg_loop(G):
    H = G.to_undirected()

    pairs_to_process = []
    indices_in_df = [] 
    
    print("Building pair list...")
    todo_list = todo.tolist() if hasattr(todo, 'tolist') else list(todo)
    
    for i, row in enumerate(df_int.itertuples()):
        if [row[1], row[3]] in todo_list:
            pos_id = f"jie:position_{int(row[1])}"
            raw_cand = str(row[3])
            cand_id = f"jie:candidate_{raw_cand}"
            
            pairs_to_process.append((pos_id, cand_id))
            indices_in_df.append(i)
    
    print(f"Found {len(pairs_to_process)} pairs to process.")

    print("Building safe graph and precomputing components...")
    blacklist_set = {'xsd:integer', 'xsd:string', 'xsd:decimal', '0', '1', 'none'}
    degrees = dict(H.degree())
    
    critical_nodes = set()
    for p in pairs_to_process:
        critical_nodes.add(p[0])
        critical_nodes.add(p[1])
    
    safe_nodes = {
        n for n in H.nodes() 
        if (n in critical_nodes) or 
           (str(n).lower() not in blacklist_set and degrees.get(n, 0) < 2500)
    }
    
    H_safe = H.subgraph(safe_nodes)
    
    print("Mapping connected components...")
    node_to_comp = {}
    for comp_id, comp_nodes in enumerate(nx.connected_components(H_safe)):
        for n in comp_nodes:
            node_to_comp[n] = comp_id

    print("Starting extraction...")
    
    sgs = {
        "cvid": [], "vacancy": [], "response": [], "graph": [],
        "path_exists": [], "num_nodes": [], "num_edges": [], 
        "density": [], "avg_degree": [], "avg_clustering": [], 
        "shortest_path_length": []
    }
    
    for i, pair in enumerate(tqdm(pairs_to_process, desc="Extracting Subgraphs")):
        idx_in_df = indices_in_df[i]
        row = df_int.iloc[idx_in_df]

        source = pair[0]
        target = pair[1]
        
        graph, metrics = extract_subgraphs(G, H_safe, node_to_comp, degrees, source, target)
        
        sgs["cvid"].append(row.cvid)
        sgs["vacancy"].append(row.humanjobid)
        sgs["response"].append(row.response)
        
        if graph is not None:
            sgs["graph"].append(nx.node_link_data(graph))
        else:
            sgs["graph"].append(None)
            
        for key in metrics:
            sgs[key].append(metrics[key])
            
        if len(sgs["path_exists"]) > 0:
            print(f"Valid Path Ratio: {np.mean(sgs['path_exists']):.2%}", end="\r")

    print("\nDone! All subgraphs processed.")

    return sgs

In [51]:
# List to hold the aggregated data for each combination
summary_results = []

for isco in [True, False]:
    for model in ["qwen", "gemma", "llama"]:
        for prompt in ["structured", "semi-structured", "unstructured"]:
            print(f"\n--- Processing: Model={model}, Prompt={prompt}, ISCO={isco} ---")
            
            G = load_KG(model, prompt, isco=isco)
            sgs = sg_loop(G)
            
            df_sgs = pd.DataFrame(sgs)
            filename = f"subgraphs_{model}_{prompt}{'_isco' if isco else ''}.xlsx"
            df_sgs.to_excel(filename, index=False)
            
            # We only calculate topological averages for pairs that actually connected
            valid_graphs = df_sgs[df_sgs["path_exists"] == True]
            
            summary_row = {
                "Model": model,
                "Prompt_Type": prompt,
                "ISCO": isco,
                "Total_Pairs": len(df_sgs),
                "Path_Found_Ratio": df_sgs["path_exists"].mean(),
            }
            
            if not valid_graphs.empty:
                summary_row.update({
                    "Avg_Nodes": valid_graphs["num_nodes"].mean(),
                    "Avg_Edges": valid_graphs["num_edges"].mean(),
                    "Avg_Density": valid_graphs["density"].mean(),
                    "Avg_Degree": valid_graphs["avg_degree"].mean(),
                    "Avg_Clustering": valid_graphs["avg_clustering"].mean(),
                    "Avg_Path_Length": valid_graphs["shortest_path_length"].mean(),
                })
            else:
                # Fallback if a combination completely fails
                summary_row.update({
                    "Avg_Nodes": 0, "Avg_Edges": 0, "Avg_Density": 0,
                    "Avg_Degree": 0, "Avg_Clustering": 0, "Avg_Path_Length": 0
                })
                
            summary_results.append(summary_row)

# 4. Generate the final master overview
df_overview = pd.DataFrame(summary_results)

# Display it in your notebook
print("\n=== FINAL METRICS OVERVIEW ===")
display(df_overview)


--- Processing: Model=qwen, Prompt=structured, ISCO=True ---
Loading qwen structured True


  0%|          | 0/2697812 [00:00<?, ?it/s]

  0%|          | 0/365821 [00:00<?, ?it/s]

Building pair list...
Found 24057 pairs to process.
Building safe graph and precomputing components...
Mapping connected components...
Starting extraction...


Extracting Subgraphs:   0%|          | 0/24057 [00:00<?, ?it/s]

...

  0%|          | 0/509404 [00:00<?, ?it/s]

Building pair list...
Found 24057 pairs to process.
Building safe graph and precomputing components...
Mapping connected components...
Starting extraction...


Extracting Subgraphs:   0%|          | 0/24057 [00:00<?, ?it/s]

Valid Path Ratio: 66.11%%
Done! All subgraphs processed.

=== FINAL METRICS OVERVIEW ===


,Model,Prompt_Type,ISCO,Total_Pairs,Path_Found_Ratio,Avg_Nodes,Avg_Edges,Avg_Density,Avg_Degree,Avg_Clustering,Avg_Path_Length
0,qwen,structured,True,24057,0.740907,15.893795,16.759762,0.147140,2.107019,0.168202,3.845209
1,qwen,semi-structured,True,24057,0.780064,15.823884,16.757860,0.148278,2.113782,0.169430,3.771608
2,qwen,unstructured,True,24057,0.757243,16.563100,16.919855,0.136025,2.041836,0.145037,4.280123
3,gemma,structured,True,24057,0.799767,15.533056,16.346518,0.150074,2.101760,0.167496,3.727599
4,gemma,semi-structured,True,24057,0.795070,15.346578,16.218801,0.153216,2.111103,0.174496,3.678883
5,gemma,unstructured,True,24057,0.760693,15.164918,15.871366,0.153380,2.091334,0.176669,3.750765
6,llama,structured,True,24057,0.664547,15.306374,16.037593,0.151384,2.091544,0.170144,3.800338
7,llama,semi-structured,True,24057,0.672860,15.249583,16.019769,0.152133,2.096751,0.174354,3.773646
8,llama,unstructured,True,24057,0.665336,15.717793,16.400662,0.146767,2.083320,0.178594,4.138448
9,qwen,structured,False,24057,0.737083,15.925502,16.798951,0.146848,2.107779,0.167764,3.849368


In [52]:
df_overview = pd.DataFrame(summary_results)
df_overview

,Model,Prompt_Type,ISCO,Total_Pairs,Path_Found_Ratio,Avg_Nodes,Avg_Edges,Avg_Density,Avg_Degree,Avg_Clustering,Avg_Path_Length
0,qwen,structured,True,24057,0.740907,15.893795,16.759762,0.147140,2.107019,0.168202,3.845209
1,qwen,semi-structured,True,24057,0.780064,15.823884,16.757860,0.148278,2.113782,0.169430,3.771608
2,qwen,unstructured,True,24057,0.757243,16.563100,16.919855,0.136025,2.041836,0.145037,4.280123
3,gemma,structured,True,24057,0.799767,15.533056,16.346518,0.150074,2.101760,0.167496,3.727599
4,gemma,semi-structured,True,24057,0.795070,15.346578,16.218801,0.153216,2.111103,0.174496,3.678883
5,gemma,unstructured,True,24057,0.760693,15.164918,15.871366,0.153380,2.091334,0.176669,3.750765
6,llama,structured,True,24057,0.664547,15.306374,16.037593,0.151384,2.091544,0.170144,3.800338
7,llama,semi-structured,True,24057,0.672860,15.249583,16.019769,0.152133,2.096751,0.174354,3.773646
8,llama,unstructured,True,24057,0.665336,15.717793,16.400662,0.146767,2.083320,0.178594,4.138448
9,qwen,structured,False,24057,0.737083,15.925502,16.798951,0.146848,2.107779,0.167764,3.849368


In [ ]:
def plot_clean_graph(G):
    plt.figure(figsize=(10, 8))
    
    # 1. REMOVE SELF-LOOPS FROM THE PLOT ONLY
    # We don't want to see them, even if they exist in the data
    display_G = G.copy()
    display_G.remove_edges_from(nx.selfloop_edges(display_G))
    
    # 2. POSITIONAL LAYOUT
    # Spring layout is good, but 'kamada_kawai' is better for seeing paths
    pos = nx.kamada_kawai_layout(display_G)
    
    # 3. DRAW
    nx.draw_networkx_nodes(display_G, pos, node_size=500, node_color='#1f78b4')
    nx.draw_networkx_edges(display_G, pos, alpha=0.5, edge_color='gray', arrows=True)
    
    # 4. CLEAN LABELS
    # Shorten the long jie: labels so they don't overlap
    labels = {n: str(n).replace("jie:", "").replace("_jie_", "").split('/')[-1][:20] for n in display_G.nodes()}
    nx.draw_networkx_labels(display_G, pos, labels, font_size=8)
    
    plt.axis('off')
    plt.show()

In [ ]:
test_pair = pairs_to_process[0]
print(f"Testing extraction for: {test_pair}")

# Run the extraction manually (sequential)
test_graph = extract_balanced_subgraph_sequential(
    H, 
    G, 
    test_pair[0], 
    test_pair[1], 
)

if test_graph:
    print(f"Nodes: {test_graph.number_of_nodes()}, Edges: {test_graph.number_of_edges()}")
    # Use your plotting function from before
    plot_clean_graph(test_graph)
else:
    print(test_graph)
    print("Extraction failed for this pair.")